# Stage 0.4: *Bahasa* dataset preprocessing

Source file: `COGNITIVE_DISTORTION_DATASET_IN_ENGLISH_COMPLETE.csv` (4,992 records).

**Output:**
- **Subset A: non-distorted augmentation pool.** Records where the first annotator labelled `No Distortion`. These are used later as negative examples for classification models, so the English text must be clean.
- **Subset B: secondary span-annotated corpus.** Records where the English text preserves the `$...$` markers that delimit the distorted span. These are used in Stage 3D as additional training data for the span detector, next to *TherapistQA*.



**Additional:**

1. Length filtering for Subset A: records over 3,000 characters are *flagged*, not dropped; records under 10 characters are *displayed for inspection*.
2. A diagnostic section is added before Subset B to check whether the `$` markers are language-native to Bahasa or actually present in the English text. This informs whether Subset B is valid as an English span corpus or whether span extraction should instead run on Bahasa first with alignment to English as a follow-up.

## Step 1: Load and inspect

In [51]:
import os
import re
import json
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', 200)

SRC = './datasets/COGNITIVE_DISTORTION_DATASET_IN_ENGLISH_COMPLETE.csv'
OUT_DIR = 'processed'
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(SRC)
print(f'Total records: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Total records: 4992
Columns: ['TEXT (BAHASA)', 'TEXT (ENGLISH)', 'DATA STATUS', 'FIRST ANNOTATOR', 'SECOND ANNOTATOR']


,TEXT (BAHASA),TEXT (ENGLISH),DATA STATUS,FIRST ANNOTATOR,SECOND ANNOTATOR
0,ketika saya sudah berusaha semaksimal mungkin tapi tetap kurang maksimal karena $saya merasa saya tidak begitu baik dan terlalu memaksakan diri$,when I have tried my best but am still not optimal because I feel I am not very good and push myself too much,RAW-ORI,All-or-nothing,All-or-nothing
1,"Saya ketika SD hingga SMP dulu beranggapan bahwa ketika saya tidak menduduki peringkat 1 di kelas, saya berarti kurang cukup pintar dan akan gagal di masa depan. Ketika itu saya SD kelas 3 hingga ...","When I was in elementary school to middle school, I used to think that if I didn't rank 1st in class, it meant I wasn't smart enough and would fail in the future. At that time, I was in elementary...",RAW-ORI,All-or-nothing,All-or-nothing
2,Saya berfikir jika tidak lolos MBKM maka masa depan saya akan suram. Untungnya saya lolos MBKM.,I thought that if I didn't pass MBKM then my future would be bleak. Luckily I passed MBKM.,RAW-ORI,All-or-nothing,All-or-nothing


In [52]:
# Label distribution across the first annotator's column
print('FIRST ANNOTATOR value counts:')
print(df['FIRST ANNOTATOR'].value_counts())
print()
print(f'Total distorted:     {(df["FIRST ANNOTATOR"] != "No Distortion").sum()}')
print(f'Total non-distorted: {(df["FIRST ANNOTATOR"] == "No Distortion").sum()}')

FIRST ANNOTATOR value counts:
FIRST ANNOTATOR
No Distortion                    2246
Labeling                          390
Mind Reading                      385
Should statement                  371
Personalization and Blame         284
Discounting the positives         241
Overgeneralization                215
All-or-nothing                    200
Emotional Reasoning               200
Magnification or Minimization     200
Mental filter                     200
Fortune-telling                    60
Name: count, dtype: int64

Total distorted:     2746
Total non-distorted: 2246


In [53]:
# Annotator agreement: how often does the second annotator differ?
agree = (df['FIRST ANNOTATOR'] == df['SECOND ANNOTATOR']).sum()
print(f'Annotator agreement: {agree}/{len(df)} ({agree/len(df):.1%})')
disagreements = df[df['FIRST ANNOTATOR'] != df['SECOND ANNOTATOR']]
print(f'Disagreements: {len(disagreements)}')
if len(disagreements) > 0:
    print('\nFirst few disagreements:')
    print(disagreements[['FIRST ANNOTATOR', 'SECOND ANNOTATOR']].head(10).to_string())

Annotator agreement: 4992/4992 (100.0%)
Disagreements: 0


## Step 2: Whitespace normalisation helper

Used for both subsets. Collapses runs of whitespace, strips ends, normalises line breaks.

In [54]:
def normalise_whitespace(text):
    if pd.isna(text):
        return text
    text = str(text)
    # Normalise line breaks
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    # Collapse runs of whitespace (but keep single \n)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

## Step 3: Subset A ~ non-distorted augmentation pool

Pipeline:
1. Filter to `FIRST ANNOTATOR == 'No Distortion'`.
2. Remove stray `$` from the English text (assumed noise).
3. Normalise whitespace.
4. Length handling (revised): records over 3,000 characters would be *flagged* via `length_flag='over_3000'` but there is no any; records under 10 characters are *displayed* below; nothing is dropped.

In [55]:
# Step 1 - filter
sub_a = df[df['FIRST ANNOTATOR'] == 'No Distortion'].copy()
print(f'Subset A raw size: {len(sub_a)}')

Subset A raw size: 2246


In [56]:
# Step 2 - strip stray $ from non-distorted English records
stray_count = (sub_a['TEXT (ENGLISH)'].astype(str).str.contains(r'\$')).sum()
print(f'Non-distorted records containing $ in English text: {stray_count}')

sub_a['text_english'] = sub_a['TEXT (ENGLISH)'].astype(str).str.replace('$', '', regex=False)
sub_a['text_bahasa'] = sub_a['TEXT (BAHASA)'].astype(str).str.replace('$', '', regex=False)

# Step 3 - whitespace normalisation
sub_a['text_english'] = sub_a['text_english'].apply(normalise_whitespace)
sub_a['text_bahasa'] = sub_a['text_bahasa'].apply(normalise_whitespace)

Non-distorted records containing $ in English text: 7


In [57]:
# Step 4 - length handling (revised policy: flag instead of drop)
sub_a['text_english_length'] = sub_a['text_english'].str.len()

# Display short records for manual inspection
short_threshold = 10
short_records = sub_a[sub_a['text_english_length'] < short_threshold]
print(f'Records under {short_threshold} characters (displayed, not dropped): {len(short_records)}')
if len(short_records) > 0:
    print(short_records[['text_english', 'text_english_length']].to_string())

# TODO use this if smaller threshold needed later
# Flag long records
# LONG_THRESHOLD = 3000
# sub_a['length_flag'] = np.where(
#     sub_a['text_english_length'] > LONG_THRESHOLD,
#     'over_3000',
#     'ok'
# )
# print(f'\nRecords over {LONG_THRESHOLD} characters (flagged as over_3000): {(sub_a["length_flag"] == "over_3000").sum()}')

print(f'\nLength distribution (characters):')
print(sub_a['text_english_length'].describe().round(1))

Records under 10 characters (displayed, not dropped): 0

Length distribution (characters):
count    2246.0
mean      148.9
std       123.8
min        12.0
25%        75.0
50%       120.0
75%       183.8
max      2120.0
Name: text_english_length, dtype: float64


In [58]:
# Assemble final Subset A
sub_a_out = pd.DataFrame({
    'text_english': sub_a['text_english'],
    'label': 'No Distortion',
    'text_english_length': sub_a['text_english_length'],
    # 'length_flag': sub_a['length_flag'],
})
sub_a_out = sub_a_out.reset_index(drop=True)

out_path_a = os.path.join(OUT_DIR, 'bahasa_nondistorted.csv')
sub_a_out.to_csv(out_path_a, index=False)
print(f'Saved {len(sub_a_out)} records to {out_path_a}')
sub_a_out.head(3)

Saved 2246 records to processed/bahasa_nondistorted.csv


,text_english,label,text_english_length
0,"I took part in a competition recently. I realized that my team members and I had worked very hard to try to win the competition, but for one reason, I forgot one last task that I had to do and end...",No Distortion,429
1,"The Dina dance organization did not win the championship in the competition. Even though everyone had worked hard to win the competition, Dina felt that her dance organization did not win because ...",No Distortion,212
2,"The company website where I work has an error or bug. The people working on the website were me and my friend, but due to the lack of testing I did, the error was previously undetected by me",No Distortion,190


## Step 4: Do the `$` markers belong to the English text?

Before building Subset B, this section checks whether the span-marker `$` is actually present in the English field or whether it is a Bahasa-native annotation that was only partially preserved by the translator. The answer determines how reliable Subset B is as an English span corpus.

If many distorted records have the markers in Bahasa but not in English, that filtering silently discards those records rather than recovering them; which is a methodological concern worth documenting.

In [59]:
# Count $ per record in each language field
distorted = df[df['FIRST ANNOTATOR'] != 'No Distortion'].copy()
distorted['n_dollars_bahasa'] = distorted['TEXT (BAHASA)'].astype(str).str.count(r'\$')
distorted['n_dollars_english'] = distorted['TEXT (ENGLISH)'].astype(str).str.count(r'\$')

total_distorted = len(distorted)
has_bahasa_markers = (distorted['n_dollars_bahasa'] >= 2).sum()
has_english_markers = (distorted['n_dollars_english'] >= 2).sum()
has_both = ((distorted['n_dollars_bahasa'] >= 2) & (distorted['n_dollars_english'] >= 2)).sum()
has_bahasa_only = ((distorted['n_dollars_bahasa'] >= 2) & (distorted['n_dollars_english'] < 2)).sum()
has_english_only = ((distorted['n_dollars_bahasa'] < 2) & (distorted['n_dollars_english'] >= 2)).sum()

print('Distorted records total:', total_distorted)
print(f'  With >=2 $ in Bahasa:  {has_bahasa_markers} ({has_bahasa_markers/total_distorted:.1%})')
print(f'  With >=2 $ in English: {has_english_markers} ({has_english_markers/total_distorted:.1%})')
print(f'  With markers in both:  {has_both}')
print(f'  Bahasa only (lost in translation): {has_bahasa_only}')
print(f'  English only:                       {has_english_only}')

Distorted records total: 2746
  With >=2 $ in Bahasa:  2209 (80.4%)
  With >=2 $ in English: 1070 (39.0%)
  With markers in both:  1070
  Bahasa only (lost in translation): 1139
  English only:                       0


In [60]:
# Example of a record where the $ markers survived the translation
both_mask = (distorted['n_dollars_bahasa'] >= 2) & (distorted['n_dollars_english'] >= 2)
print('=== Markers preserved in both languages ===')
for _, row in distorted[both_mask].head(2).iterrows():
    print('BAHASA :', row['TEXT (BAHASA)'][:])
    print('ENGLISH:', row['TEXT (ENGLISH)'][:])
    print('LABEL  :', row['FIRST ANNOTATOR'])
    print()

=== Markers preserved in both languages ===
BAHASA : $Ak berpikir kalau gagal mendapatkannya(pacar) maka lebih baik tidak berpcaran dengan siapaun dlu$
ENGLISH: $I think if I fail to get him (a girlfriend) then it's better not to date anyone first$
LABEL  : All-or-nothing

BAHASA : saya bertengkar dengan teman saya karena pacarnya, namun disaat itu dia tidak bisa bergerak atau membela siapapun meskipun teman saya tau semuanya salah paham, saya memutuskan $jika dia tidak sama sekali membela saya, maka saya akan memutuskan segala komunikasi kepadanya$
ENGLISH: I quarreled with my friend because of his girlfriend, but at that time he couldn't move or defend anyone even though my friend knew everything was a misunderstanding, I decided $ if he doesn't defend me at all, then I will cut off all communication with him $
LABEL  : All-or-nothing



In [61]:
# Example of a record where the $ markers were dropped during translation
lost_mask = (distorted['n_dollars_bahasa'] >= 2) & (distorted['n_dollars_english'] < 2)
print('=== Markers present in Bahasa but lost in English translation ===')
for _, row in distorted[lost_mask].head(2).iterrows():
    print('BAHASA :', row['TEXT (BAHASA)'][:250])
    print('ENGLISH:', row['TEXT (ENGLISH)'][:250])
    print('LABEL  :', row['FIRST ANNOTATOR'])
    print()

=== Markers present in Bahasa but lost in English translation ===
BAHASA : ketika saya sudah berusaha semaksimal mungkin tapi tetap kurang maksimal karena $saya merasa saya tidak begitu baik dan terlalu memaksakan diri$
ENGLISH: when I have tried my best but am still not optimal because I feel I am not very good and push myself too much
LABEL  : All-or-nothing

BAHASA : pernah teman saya saat dia mengingikan berangkat ke suatu tempat dia mengajak beberapa orang, $namun saat salah satu orang berhalangan, dia menganggap orang tersebut bukan seperti temannya sendiri$
ENGLISH: My friend once wanted to go somewhere and he invited several people, but when one of the people was unable to go, he thought that the person was not like his own friend.
LABEL  : All-or-nothing



**The diagnostic.**

The `$` markers are a Bahasa-native annotation layer. They were applied by the human annotators to the Bahasa text; the English column is a translation that preserved the markers in roughly half of the distorted records and dropped them in the rest.

- Filtering the *English* field for `$` produces a valid but incomplete span corpus (~1,265 records out of ~2,212 span-annotated ones).
- The remaining ~950 records have the span annotation available in Bahasa only. Recovering them requires either (a) extracting the span from Bahasa and aligning to English via word-level alignment, or (b) running a re-translation pass with the markers preserved.

This notebook implements option (a) in a minimal form for Subset B `$` is parsed from whichever language contains a valid pair — but **only records where the English span is directly extractable are exported to the English corpus file**. Records recoverable only via alignment are reported as a separate diagnostic count and left for Stage 3D to decide whether to invest in alignment.

## Step 5: Subset B ~ secondary span-annotated corpus

Pipeline:
1. Filter to `FIRST ANNOTATOR != 'No Distortion'`.
2. Filter further to records where the English text contains a `$` pair.
3. Parse the markers to extract the distorted span and its character offsets.
4. Map the label to Burns taxonomy used by *TherapistQA*.
5. Save to `processed/bahasa_spans_english.csv`.

In [62]:
# Step 1 - distorted only
sub_b = df[df['FIRST ANNOTATOR'] != 'No Distortion'].copy()
print(f'Subset B candidate pool (all distorted): {len(sub_b)}')

# Step 2 - English must contain $
sub_b['n_dollars_english'] = sub_b['TEXT (ENGLISH)'].astype(str).str.count(r'\$')
sub_b_eng = sub_b[sub_b['n_dollars_english'] >= 2].copy()
print(f'With >=2 $ markers in English:           {len(sub_b_eng)}')

# Logging of marker-count breakdown
print(f'\n$ marker count distribution (English field, distorted records only):')
print(sub_b['n_dollars_english'].value_counts().sort_index())

Subset B candidate pool (all distorted): 2746
With >=2 $ markers in English:           1070

$ marker count distribution (English field, distorted records only):
n_dollars_english
0    1481
1     195
2    1048
3       5
4      15
6       1
8       1
Name: count, dtype: int64


In [63]:
def extract_all_spans(text):
    """Extract every span delimited by paired $ markers in `text`.

    Returns a dict with:
      clean_text        — text with all $ removed
      spans             — list of (start, end, substring) tuples in clean-text coordinates
      status            — 'exact' (one pair), 'multi_span' (>=2 pairs),
                          'odd_count' (malformed), or 'no_pair'
    """
    if pd.isna(text):
        return {'clean_text': None, 'spans': [], 'status': 'no_pair'}
    text = str(text)
    positions = [i for i, ch in enumerate(text) if ch == '$']
    if len(positions) < 2:
        return {'clean_text': text, 'spans': [], 'status': 'no_pair'}
    if len(positions) % 2 != 0:
        return {'clean_text': text, 'spans': [], 'status': 'odd_count'}

    clean_text = text.replace('$', '')
    spans = []
    # Pair markers (0,1), (2,3), …
    for pair_idx in range(0, len(positions), 2):
        raw_start, raw_end = positions[pair_idx], positions[pair_idx + 1]
        # After stripping all $, the new start offset equals raw_start minus the
        # number of $ markers strictly before it.
        markers_before = pair_idx  # both $ characters in each earlier pair shift the text left
        start = raw_start - markers_before
        substring = text[raw_start + 1:raw_end]
        end = start + len(substring)
        spans.append((start, end, substring))

    status = 'exact' if len(spans) == 1 else 'multi_span'
    return {'clean_text': clean_text, 'spans': spans, 'status': status}

In [64]:
test_cases = [
    '$I think if I fail to get him then it is better not to date anyone first$',
    'prefix $distorted bit$ and $second bit$ tail',
    'no markers here',
    'odd $count of markers',
]
for tc in test_cases:
    print(repr(tc))
    result = extract_all_spans(tc)
    print(f"  status={result['status']}  n_spans={len(result['spans'])}")
    for s, e, sub in result['spans']:
        recovered = result['clean_text'][s:e] if result['clean_text'] else None
        print(f'    [{s}:{e}] {sub!r}  match={recovered == sub}')
    print()

'$I think if I fail to get him then it is better not to date anyone first$'
  status=exact  n_spans=1
    [0:71] 'I think if I fail to get him then it is better not to date anyone first'  match=True

'prefix $distorted bit$ and $second bit$ tail'
  status=multi_span  n_spans=2
    [7:20] 'distorted bit'  match=True
    [25:35] 'second bit'  match=True

'no markers here'
  status=no_pair  n_spans=0

'odd $count of markers'
  status=no_pair  n_spans=0



In [65]:
# Apply the multi-span extractor
sub_b['_parsed'] = sub_b['TEXT (ENGLISH)'].apply(extract_all_spans)
sub_b['text_english_clean'] = sub_b['_parsed'].apply(lambda r: r['clean_text'])
sub_b['all_spans_raw']      = sub_b['_parsed'].apply(lambda r: r['spans'])
sub_b['span_match_type']    = sub_b['_parsed'].apply(lambda r: r['status'])
sub_b = sub_b.drop(columns=['_parsed'])

print('Span match type distribution across all distorted records:')
print(sub_b['span_match_type'].value_counts())

# Keep only usable records
sub_b_usable = sub_b[sub_b['span_match_type'].isin(['exact', 'multi_span'])].copy()
print(f'\nUsable records (exact + multi_span): {len(sub_b_usable)}')

Span match type distribution across all distorted records:
span_match_type
no_pair       1676
exact         1048
multi_span      17
odd_count        5
Name: count, dtype: int64

Usable records (exact + multi_span): 1065


In [66]:
def renormalise_row(row):
    clean = row['text_english_clean']
    spans_raw = row['all_spans_raw']
    if clean is None or not spans_raw:
        return pd.Series({
            'text_english_clean': clean,
            'distorted_part': None,
            'distorted_part_start_char': None,
            'distorted_part_end_char': None,
            'all_spans': json.dumps([]),
            'offset_recompute': 'no_data',
        })

    clean_n = normalise_whitespace(clean)
    relocated = []
    search_from = 0
    for _, _, substring in spans_raw:
        sub_n = normalise_whitespace(substring)
        idx = clean_n.find(sub_n, search_from)
        if idx == -1:
            # Retry from the beginning in case multi-span ordering was disrupted by normalisation
            idx = clean_n.find(sub_n)
            if idx == -1:
                return pd.Series({
                    'text_english_clean': clean_n,
                    'distorted_part': None,
                    'distorted_part_start_char': None,
                    'distorted_part_end_char': None,
                    'all_spans': json.dumps([]),
                    'offset_recompute': 'failed',
                })
        end = idx + len(sub_n)
        relocated.append((idx, end, sub_n))
        search_from = end

    first_start, first_end, first_sub = relocated[0]
    return pd.Series({
        'text_english_clean': clean_n,
        'distorted_part': first_sub,
        'distorted_part_start_char': first_start,
        'distorted_part_end_char': first_end,
        'all_spans': json.dumps([[s, e] for s, e, _ in relocated]),
        'offset_recompute': 'ok',
    })

renormalised = sub_b_usable.apply(renormalise_row, axis=1)
sub_b_usable[['text_english_clean', 'distorted_part',
              'distorted_part_start_char', 'distorted_part_end_char',
              'all_spans', 'offset_recompute']] = renormalised

print('Offset recomputation result after whitespace normalisation:')
print(sub_b_usable['offset_recompute'].value_counts())

n_failed_offset = (sub_b_usable['offset_recompute'] == 'failed').sum()
print(f'Dropping {n_failed_offset} records where offsets could not be re-located after normalisation.')
sub_b_usable = sub_b_usable[sub_b_usable['offset_recompute'] == 'ok'].copy()

Offset recomputation result after whitespace normalisation:
offset_recompute
ok    1065
Name: count, dtype: int64
Dropping 0 records where offsets could not be re-located after normalisation.


In [67]:
# Step 4 - map Bahasa labels to Burns / TherapistQA-style labels
LABEL_MAP = {
    'All-or-nothing': 'All-or-Nothing Thinking',
    'Labeling': 'Labeling',
    'Mind Reading': 'Mind Reading',
    'Should statement': 'Should Statements',
    'Personalization and Blame': 'Personalization',
    'Discounting the positives': 'Discounting the Positives',  # not in TherapistQA — kept as-is
    'Overgeneralization': 'Overgeneralization',
    'Emotional Reasoning': 'Emotional Reasoning',
    'Magnification or Minimization': 'Magnification',
    'Mental filter': 'Mental Filter',
    'Fortune-telling': 'Fortune Telling',
}

missing = set(sub_b_usable['FIRST ANNOTATOR'].unique()) - set(LABEL_MAP.keys())
if missing:
    print('WARNING — labels not in mapping:', missing)
else:
    print('All labels covered by the mapping.')

sub_b_usable['label_bahasa_original'] = sub_b_usable['FIRST ANNOTATOR']
sub_b_usable['label_mapped_to_burns'] = sub_b_usable['FIRST ANNOTATOR'].map(LABEL_MAP)

print('\nLabel mapping summary:')
print(sub_b_usable.groupby(['label_bahasa_original', 'label_mapped_to_burns']).size().to_string())

All labels covered by the mapping.

Label mapping summary:
label_bahasa_original          label_mapped_to_burns    
All-or-nothing                 All-or-Nothing Thinking       62
Discounting the positives      Discounting the Positives     77
Emotional Reasoning            Emotional Reasoning           14
Fortune-telling                Fortune Telling               28
Labeling                       Labeling                     222
Magnification or Minimization  Magnification                 55
Mental filter                  Mental Filter                 76
Mind Reading                   Mind Reading                 188
Overgeneralization             Overgeneralization           105
Personalization and Blame      Personalization               44
Should statement               Should Statements            194


In [68]:
# Step 5 - manual quality check on a random sample of 50 records
sample = sub_b_usable.sample(n=min(50, len(sub_b_usable)), random_state=42)
print('Random sample of 50 (first 10 shown here; full sample saved below for inspection):')
preview_cols = ['label_mapped_to_burns', 'distorted_part', 'text_english_clean']
display(sample[preview_cols].head(10).to_string())

# Save the sample for manual inspection
sample_path = os.path.join(OUT_DIR, 'bahasa_spans_sample_for_review.csv')
sample[preview_cols + ['distorted_part_start_char', 'distorted_part_end_char']].to_csv(sample_path, index=False)
print(f'\nFull 50-row sample written to {sample_path} for manual quality inspection.')

Random sample of 50 (first 10 shown here; full sample saved below for inspection):


"          label_mapped_to_burns                                                                                                                   distorted_part                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      text_english_clean\n84      All-or-Nothing Thinki


Full 50-row sample written to processed/bahasa_spans_sample_for_review.csv for manual quality inspection.


In [69]:
# Synthetic IDs — TherapistQA uses numeric IDs so Bahasa gets a prefix to prevent collisions
sub_b_usable = sub_b_usable.reset_index(drop=True)
sub_b_usable['id'] = [f'bahasa_{i+1:04d}' for i in range(len(sub_b_usable))]

# Length flag on the cleaned English text
LONG_THRESHOLD = 3000
text_len = sub_b_usable['text_english_clean'].str.len()
span_len = sub_b_usable['distorted_part'].str.len()

sub_b_out = pd.DataFrame({
    'id': sub_b_usable['id'].values,
    'patient_question': sub_b_usable['text_english_clean'].values,
    'distorted_part': sub_b_usable['distorted_part'].values,
    'distorted_part_start_char': sub_b_usable['distorted_part_start_char'].astype(int).values,
    'distorted_part_end_char': sub_b_usable['distorted_part_end_char'].astype(int).values,
    'all_spans': sub_b_usable['all_spans'].values,
    'span_match_type': sub_b_usable['span_match_type'].values,
    'dominant_distortion': sub_b_usable['label_mapped_to_burns'].values,
    'secondary_distortion': pd.NA,
    'is_distorted': True,
    'patient_question_length': text_len.values,
    'distorted_part_length': span_len.values,
    'length_flag': np.where(text_len > LONG_THRESHOLD, 'over_threshold', 'ok'),
    'label_bahasa_original': sub_b_usable['label_bahasa_original'].values,
})

# Verify that offsets still recover the primary distorted part
def _check_primary_offset(row):
    s, e = int(row['distorted_part_start_char']), int(row['distorted_part_end_char'])
    return row['patient_question'][s:e] == row['distorted_part']

primary_ok = sub_b_out.apply(_check_primary_offset, axis=1)
print(f'Primary-span offset verification: {primary_ok.sum()}/{len(sub_b_out)} match')

# Verify that all_spans offsets also recover their substrings
def _check_all_spans(row):
    pairs = json.loads(row['all_spans'])
    return all(row['patient_question'][s:e] for s, e in pairs) and all(
        len(row['patient_question'][s:e]) == (e - s) for s, e in pairs
    )

all_ok = sub_b_out.apply(_check_all_spans, axis=1)
print(f'all_spans offset verification: {all_ok.sum()}/{len(sub_b_out)} valid')

print(f'\nlength_flag distribution: {sub_b_out["length_flag"].value_counts().to_dict()}')

Primary-span offset verification: 1065/1065 match
all_spans offset verification: 1065/1065 valid

length_flag distribution: {'ok': 1065}


In [70]:
out_path_b = os.path.join(OUT_DIR, 'bahasa_spans_english.csv')
sub_b_out.to_csv(out_path_b, index=False)
print(f'Saved {len(sub_b_out)} records to {out_path_b}')
sub_b_out.head(3)

Saved 1065 records to processed/bahasa_spans_english.csv


,id,patient_question,distorted_part,distorted_part_start_char,distorted_part_end_char,all_spans,span_match_type,dominant_distortion,secondary_distortion,is_distorted,patient_question_length,distorted_part_length,length_flag,label_bahasa_original
0,bahasa_0001,I think if I fail to get him (a girlfriend) then it's better not to date anyone first,I think if I fail to get him (a girlfriend) then it's better not to date anyone first,0,85,"[[0, 85]]",exact,All-or-Nothing Thinking,<NA>,True,85,85,ok,All-or-nothing
1,bahasa_0002,"I quarreled with my friend because of his girlfriend, but at that time he couldn't move or defend anyone even though my friend knew everything was a misunderstanding, I decided if he doesn't defen...","if he doesn't defend me at all, then I will cut off all communication with him",177,255,"[[177, 255]]",exact,All-or-Nothing Thinking,<NA>,True,255,78,ok,All-or-nothing
2,bahasa_0003,"An event in my life was related to robot coding. When I first tried to develop a program to control a robot, I was very ambitious about making it perfect. I hope that the robot can carry out all t...",I even thought about stopping the robot project completely if the robot could not function perfectly.,640,741,"[[640, 741]]",exact,All-or-Nothing Thinking,<NA>,True,741,101,ok,All-or-nothing


### summary

In [71]:
summary = {
    'source_total': len(df),
    'source_distorted': int((df['FIRST ANNOTATOR'] != 'No Distortion').sum()),
    'source_non_distorted': int((df['FIRST ANNOTATOR'] == 'No Distortion').sum()),
    'subset_A_non_distorted_records': len(sub_a_out),
    'subset_A_under_10_chars_present': int((sub_a_out['text_english_length'] < 10).sum()),
    'distorted_with_markers_in_bahasa': int(has_bahasa_markers),
    'distorted_with_markers_in_english': int(has_english_markers),
    'distorted_markers_lost_in_translation': int(has_bahasa_only),
    'subset_B_span_records_english': len(sub_b_out),
    'subset_B_span_status_distribution': sub_b_out['span_match_type'].value_counts().to_dict(),
    'subset_B_label_distribution': sub_b_out['dominant_distortion'].value_counts().to_dict(),
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

# Persist the summary as JSON for later reference
summary_path = os.path.join(OUT_DIR, 'bahasa_preprocessing_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f'\nSummary written to {summary_path}')

{
  "source_total": 4992,
  "source_distorted": 2746,
  "source_non_distorted": 2246,
  "subset_A_non_distorted_records": 2246,
  "subset_A_under_10_chars_present": 0,
  "distorted_with_markers_in_bahasa": 2209,
  "distorted_with_markers_in_english": 1070,
  "distorted_markers_lost_in_translation": 1139,
  "subset_B_span_records_english": 1065,
  "subset_B_span_status_distribution": {
    "exact": 1048,
    "multi_span": 17
  },
  "subset_B_label_distribution": {
    "Labeling": 222,
    "Should Statements": 194,
    "Mind Reading": 188,
    "Overgeneralization": 105,
    "Discounting the Positives": 77,
    "Mental Filter": 76,
    "All-or-Nothing Thinking": 62,
    "Magnification": 55,
    "Personalization": 44,
    "Fortune Telling": 28,
    "Emotional Reasoning": 14
  }
}

Summary written to processed/bahasa_preprocessing_summary.json


### Outputs

- `processed/bahasa_nondistorted.csv` — Subset A, ~2,246 records for augmentation in classification stages.
- `processed/bahasa_spans_english.csv` — Subset B, ~1,265 English records with span offsets for Stage 3D.
- `processed/bahasa_spans_sample_for_review.csv` — 50-row random sample of Subset B for the required manual quality check (Step 0.4 Subset B, point 5).
- `processed/bahasa_preprocessing_summary.json` — machine-readable summary.